In [36]:
%pip install statsmodels


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/martanarozhnyak/py312/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
# Importing libraries

import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats
from scipy.stats import chi2_contingency

In [38]:
# Prepare data for Z-test

# Step 1: Load the csv file of Q2: Which acquisition channel converts best?

df_1 = pd.read_csv('/Users/martanarozhnyak/Desktop/ga4_project/ga4_q2.csv')

In [39]:
# Step 2: Preview the dataset

df_1

,medium,source,unique_users,total_sessions,purchasing_sessions,conversion_rate_pct
0,organic,google,103487,111523,1250,1.12
1,(none),(direct),75951,82853,1079,1.30
2,<Other>,<Other>,51037,51799,506,0.98
3,referral,<Other>,32880,34566,471,1.36
4,referral,shop.googlemerchandisestore.com,26065,28750,585,2.03
5,cpc,google,15527,15594,153,0.98
6,organic,<Other>,10095,10151,97,0.96


In [40]:
# Step 3: filter rows
organic_google_row = df_1[
    (df_1['medium'] == 'organic') &
    (df_1['source'] == 'google')
]

none_direct_row = df_1[
    (df_1['medium'] == '(none)') &
    (df_1['source'] == '(direct)')
]

# Step 4: extract the scalar values for filtered rows
# Use .iloc[0] to extract the raw scalar — without it, filtering returns
# a pandas Series (with index and metadata), which would break the z-test.
organic_google_purchasing_sessions = organic_google_row['purchasing_sessions'].iloc[0]
none_direct_purchasing_sessions = none_direct_row['purchasing_sessions'].iloc[0]
organic_google_total_sessions = organic_google_row['total_sessions'].iloc[0]
none_direct_total_sessions = none_direct_row['total_sessions'].iloc[0]

In [41]:
# Step 6: Checking the outputs
print(f"Organic/Google Total Sessions: {organic_google_total_sessions:,.0f}")
print(f"Organic/Google Purchasing Sessions: {organic_google_purchasing_sessions:,.0f}")

print(f"None/Direct Total Sessions: {none_direct_total_sessions:,.0f}")
print(f"None/Direct Purchasing Sessions: {none_direct_purchasing_sessions:,.0f}")

Organic/Google Total Sessions: 111,523
Organic/Google Purchasing Sessions: 1,250
None/Direct Total Sessions: 82,853
None/Direct Purchasing Sessions: 1,079


In [42]:
# Step 7: Set up inputs for the Z-test

count = [organic_google_purchasing_sessions, none_direct_purchasing_sessions]
nobs  = [organic_google_total_sessions, none_direct_total_sessions]

In [43]:
# Step 8: Run the Z-test

z_stat, p_value = proportions_ztest(count, nobs)

print(f"Z-statistic: {z_stat:,.2f}")
print(f"P-value: {p_value:.2e}")

Z-statistic: -3.64
P-value: 2.77e-04


In [44]:
# Step 9: Summary of Z-test

# Statistical test: Two-proportion z-test

# Question: Is the conversion-rate gap between organic/google and direct traffic
# statistically significant, or could it be random variation?

# H₀: organic_google conversion rate = direct conversion rate
# H₁: they differ (two-sided)
# α = 0.05

# Inputs:
#   organic/google: 1250 purchases out of 111,523 sessions (1.12%)
#   (none)/(direct): 1079 purchases out of  82,853 sessions (1.30%)

# Result: z = -3.64, p = 2.77e-04

# Conclusion: Since p < 0.05, we reject H₀. Direct traffic converts at a
# statistically significantly higher rate than organic Google search.
# The absolute gap is small (0.18 pp), but with ~194K combined sessions
# the test detects this small effect reliably. In relative terms, direct
# traffic generates about 16% more purchases per session than organic

In [45]:
# Prepare data for Chi-square test

# Step 1: Load the csv file of Q4: Mobile vs desktop conversion difference

df_2 = pd.read_csv('/Users/martanarozhnyak/Desktop/ga4_project/ga4_q4.csv')

In [46]:
# Step 2: Preview the dataset
df_2.head()

,device_type,unique_users,total_sessions,purchasing_sessions,conversion_rate_pct,total_purchase_revenue,avg_revenue_per_purchase
0,desktop,158917,205273,2749,1.34,208815.0,75.96
1,mobile,109195,141374,1995,1.41,146768.0,73.57
2,tablet,6250,7992,104,1.30,6582.0,63.29


In [47]:
# Step 3: Add a new column with non purchasing sessions to the dataframe

df_2['not_purchasing_sessions'] = df_2['total_sessions'] - df_2['purchasing_sessions']

In [48]:
# Step 4: Set up variable for the Chi-square test

contingency = df_2[['purchasing_sessions', 'not_purchasing_sessions']].values

In [49]:
chi2_stat, p_value, dof, expected = chi2_contingency(contingency)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of Freedom: {dof}")
print("\nExpected Frequencies:")
print(np.round(expected, 2))

Chi-Square Statistic: 3.4769
P-value: 0.1758
Degrees of Freedom: 2

Expected Frequencies:
[[2.8061300e+03 2.0246687e+05]
 [1.9326200e+03 1.3944138e+05]
 [1.0925000e+02 7.8827500e+03]]


In [50]:
# Step 5: Summary of Chi-square test

# Statistical test: Chi-Square Test of Independence

# Question: Is conversion (yes/no) independent of device type?

# H₀: device type and purchasing are independent (conversion rate is the same across devices)
# H₁: device type and purchasing are NOT independent (rates differ)
# α = 0.05

# Contingency table (purchased vs. did not purchase):
#   desktop: 2749 / 202,524
#   mobile:  1995 / 139,379
#   tablet:  104 /   7,888


# Result: chi2 = 3.48, p = 0.1758, dof = 2

# Conclusion: The p-value (0.1758) > 0.05, so we fail to reject H₀. 
# The conversion rates do look slightly different across devices
# (desktop 1.34%, mobile 1.41%, tablet 1.30%), but the gaps are small enough
# that they could easily be random rather than a real device effect.
#
# Even with ~355K total sessions, the test doesn't find enough evidence to
# say that device type meaningfully changes whether a session ends in a
# purchase. For this dataset, device choice doesn't predict conversion.